# Loan Default Prediction

**Author:** VEDANT MATHUR

This notebook develops and compares machine learning classification models for predicting whether a loan will be fully repaid.

### Objective
Use borrower credit and financial information to identify repayment risk and support data-driven lending decisions.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

sns.set_theme(style="whitegrid")

## 2. Load the Dataset

Place `loan_borrower_data.csv` in the same folder as this notebook.

In [ ]:
DATA_PATH = "loan_borrower_data.csv"
loans = pd.read_csv(DATA_PATH)

print("Dataset shape:", loans.shape)
display(loans.head())

## 3. Initial Data Checks

In [ ]:
print(loans.info())
print("\nMissing values:")
display(loans.isnull().sum())

print("\nDuplicate rows:", loans.duplicated().sum())

In [ ]:
display(loans.describe(include="all").T)

## 4. Exploratory Data Analysis

The target variable is `not.fully.paid`. We inspect its distribution and several credit-related variables.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=loans, x="not.fully.paid")
plt.title("Loan Repayment Outcome")
plt.xlabel("Not Fully Paid")
plt.ylabel("Number of Loans")
plt.show()

print(loans["not.fully.paid"].value_counts(normalize=True).rename("proportion"))

In [ ]:
numeric_features = [
    "int.rate", "installment", "log.annual.inc", "dti",
    "fico", "days.with.cr.line", "revol.bal", "revol.util",
    "inq.last.6mths", "delinq.2yrs", "pub.rec"
]

for feature in ["fico", "dti", "int.rate", "revol.util"]:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=loans, x="not.fully.paid", y=feature)
    plt.title(f"{feature} vs Loan Outcome")
    plt.show()

## 5. Prepare Features and Target

In [ ]:
target = "not.fully.paid"

X = loans.drop(columns=[target])
y = loans[target]

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

## 6. Train/Test Split

Stratification is used because the target classes are not evenly distributed.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

## 7. Model Selection

Three tree-based classification models are compared:

1. Decision Tree
2. Random Forest
3. Gradient Boosting

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(
        random_state=42,
        class_weight="balanced"
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42,
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3
    )
}

pipelines = {
    name: Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    for name, model in models.items()
}

## 8. Cross-Validation

F1-score is used alongside other metrics because loan-default datasets can have class imbalance.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []

for name, pipeline in pipelines.items():
    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="f1"
    )
    cv_results.append({
        "Model": name,
        "Mean CV F1": scores.mean(),
        "Std CV F1": scores.std()
    })

cv_results = pd.DataFrame(cv_results).sort_values("Mean CV F1", ascending=False)
display(cv_results)

## 9. Test Set Evaluation

In [ ]:
evaluation = []

for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    evaluation.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1 Score": f1_score(y_test, predictions, zero_division=0)
    })

evaluation = pd.DataFrame(evaluation).sort_values("F1 Score", ascending=False)
display(evaluation)

## 10. Confusion Matrices

In [ ]:
for name, pipeline in pipelines.items():
    predictions = pipeline.predict(X_test)
    cm = confusion_matrix(y_test, predictions)

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Fully Paid", "Not Fully Paid"],
        yticklabels=["Fully Paid", "Not Fully Paid"]
    )
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

## 11. Best Model

The model with the highest test-set F1 score is selected for this comparison. The final classification report provides a detailed view of its performance.

In [ ]:
best_model_name = evaluation.iloc[0]["Model"]
best_model = pipelines[best_model_name]

best_predictions = best_model.predict(X_test)

print("Best model:", best_model_name)
print()
print(classification_report(
    y_test,
    best_predictions,
    target_names=["Fully Paid", "Not Fully Paid"],
    zero_division=0
))

## 12. Conclusion

The project compares multiple machine learning approaches for loan repayment-risk prediction.

Gradient Boosting is included as the primary candidate because it can capture non-linear relationships between borrower characteristics and repayment outcomes. The final model should be selected based on the evaluation metrics and the business cost of different types of classification errors.

This workflow can be extended with hyperparameter tuning, class-imbalance techniques, explainable AI, and model deployment.